In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructField,
    StructType,
    StringType,
    IntegerType,
    DoubleType,
    LongType
)


# Define Source Schema 

schema=StructType([
    StructField('InvoiceNo',StringType(),True),
    StructField('StockCode',StringType(),True),
    StructField('Description',StringType(),True),
    StructField('Quantity',IntegerType(),True),
    StructField('InvoiceDate',StringType(),True),
    StructField('UnitPrice',DoubleType(),True),
    StructField('CustomerID',DoubleType(),True),
    StructField('Country',StringType(),True)
])

input_path = "/Volumes/data_migration_project/default/online_retail/online_retail.csv"


df=(
    spark.read.csv(
        input_path,
        header=True,
        encoding='ISO-8859-1',
        schema=schema
    )
)


print('============= SOURCE DATA ===============')

print('Initital record count: ', df.count())
df.printSchema()

df.show(10,truncate=False)

# Remove exact duplicates

df = df.dropDuplicates()
print('After Removing duplicates: ',df.count())




# convert data types 

df=(
    df
    .withColumn(
        'InvoiceDate',
        F.to_timestamp(
            'InvoiceDate',
            'yyyy-MM-dd HH:mm:ss'
        )
    )

    .withColumn(
        'CustomerID',
        F.col('CustomerID').cast(LongType())
    )
)
print(f'Coverted data types: ')
df.show()

# Create Revenue

df=df.withColumn(
    'Renvenue',
    F.col('Quantity')*F.col('UnitPrice')
)


# Identify Transaction Type

df=df.withColumn(
    "TransactionType",
    F.when(
        F.col('InvoiceNo').startswith('C'),
        'RETURN'
    ).otherwise('SALE')
)



# Create Date Columns

df=(
    df
    .withColumn('Year',F.year('InvoiceDate'))
    .withColumn('Month',F.month('InvoiceDate'))
    .withColumn('Day',F.day('InvoiceDate'))
)



# Data quality Rules


valid_condition=(
    F.col('Quantity').isNotNull() 
    & F.col('UnitPrice').isNotNull()
    & F.col('InvoiceDate').isNotNull()
    & (F.col('Quantity')!=0)
    & (F.col('UnitPrice')>0)
)



valid_df=df.filter(valid_condition)


reject_df=df.filter(~valid_condition)


print('================== DATA QUALITY ==================')


print('Valid Records: ',valid_df.count())

print('Reject Records: ',reject_df.count())



# Inspect Valid data

print('============== VALID DATA =================')

valid_df.show(100,truncate=False)


# Write Sliver Locally

silver_path='/Workspace/Users/poosalasaicharan7@gmail.com/Data_Migration_Project/output/sliver_path'

(
    valid_df
    .write
    .mode('overwrite')
    .partitionBy('Year','Month')
    .parquet(silver_path)
)


# Inspect Reject data

reject_path='/Workspace/Users/poosalasaicharan7@gmail.com/Data_Migration_Project/output/reject_path'
(
    reject_df
    .write
    .mode('overwrite')
    .parquet(reject_path)
)


# Transaction summary

print('============== TRANSACTION SUMMARY ==================')


df.groupBy('TransactionType').count().show()

valid_df.select(
    F.sum('Revenue').alias('TotalRevenue')
).show()



# Data Range


print('============= DATE RANGE ===============')

df.select(
    F.min('InvoiceDate').alias('MinDate'),
    F.max('InvoiceDate').alias('maxDate')
).show()


spark.stop()